# Markowitz Portfolio Optimization
**Dimitar Zhelezov**, Master's in Mathematical Modelling in Economics, FMI, SU

## 1. Introduction

How should an investor split their money between several assets? In 1952, Harry Markowitz answered this question by treating it as a mathematical optimization problem: investors want high expected return and low risk, and these two goals pull against each other. For every target level of return, there is exactly one portfolio with the lowest possible risk. The set of all such portfolios forms the *efficient frontier*, and Markowitz's work became the foundation of modern portfolio theory.

In this project, I shall derive the efficient frontier mathematically and then compute it in Python on real market data (10 assets). I will operate within the scope of the course material (linear algebra, calculus, probability and statistics) and introduce two new concepts: **Lagrange multipliers** (optimization with constraints) and the **matrix inverse**.

The notebook is organized as follows:

1. Introduction and assumptions
2. Setting up the problem
3. Why diversification works
4. The optimization problem
5. New concept: the matrix inverse
6. New concept: Lagrange multipliers
7. The global minimum-variance portfolio
8. The efficient frontier
9. Application to real data
10. Conclusion

### Assumptions
- An investor cares only about the expected return and the risk (variance) of the portfolio.
- There is a single investment period: the investor chooses weights once and holds them.
- Expected returns and covariances are known. In practice they are estimated from historical data.
- There are no transaction costs or taxes, and assets can be bought in any fraction.
- Every asset carries some risk, and no asset's returns are an exact combination of the others' (mathematically: the covariance matrix is invertible).
- Short selling is allowed (weights may be negative).
- A portfolio holds at least 2 assets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf #I use Yahoo Finance to get my data. The data consists of 10 assets. This library allows #direct access to much financial data

## 2. Setting up the problem

Let $P_{i,t}$ denote the price of asset $i$ at time $t$. The return of asset $i$ over one period is

$$
R_{i,t} = \frac{P_{i,t} - P_{i,t-1}}{P_{i,t-1}}.
$$

Before it is realized, the return is a random variable. An investor splits their wealth across $n$ assets, putting a fraction $w_i$ (the *weight*) into asset $i$, with all weights summing to one:

$$
\sum_{i=1}^{n} w_i = 1 \quad \Longleftrightarrow \quad \mathbf{1}^\top w = 1,
$$

where $\mathbf{1} = (1, \ldots, 1)^\top$ is a vector of ones. Since short selling is allowed, individual weights may be negative or larger than one. This is called the *budget constraint* (it will enter the optimization problem later).

The portfolio return is the weighted sum of the asset returns:

$$
R_p = \sum_{i=1}^{n} w_i R_i = w^\top R,
$$

where $w = (w_1, \ldots, w_n)^\top$ is the vector of weights and $R = (R_1, \ldots, R_n)^\top$ is the random vector of returns (they are random, since they are forward looking).

By linearity of expectation, the expected portfolio return is

$$
\mathbb{E}[R_p] = \mu_p = \sum_{i=1}^{n} w_i \, \mathbb{E}[R_i] = w^\top \mu,
$$

where $\mu = (\mu_1, \ldots, \mu_n)^\top$ is the vector of expected returns.

The risk of a portfolio is measured by the variance of its return. Unlike the expected return, it depends not only on each asset's own variance but also on how the assets move together (their covariances):

$$
\sigma_p^2 = \operatorname{Var}(R_p) = \sum_{i=1}^{n}\sum_{j=1}^{n} w_i w_j \sigma_{ij} = w^\top \Sigma w,
$$

where $\Sigma$ is the covariance matrix of returns, with entries $\sigma_{ij} = \operatorname{Cov}(R_i, R_j)$ and $\sigma_{ii} = \sigma_i^2$. The double sum follows from the bilinearity of covariance: $\operatorname{Var}\left(\sum_i w_i R_i\right) = \sum_i \sum_j w_i w_j \operatorname{Cov}(R_i, R_j)$.

The square root $\sigma_p$ (the standard deviation, or *volatility*) is in the same units as returns, so it is what I plot.